In [1]:
import os
import random
from pathlib import Path

import albumentations as A
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms



In [2]:
model_path = "../input/rn-wc-tta-calr/model(6).pth"  # may be unavailable
sample_sub_path = "../input/cassava-leaf-disease-classification/sample_submission.csv"
test_images_path = "../input/cassava-leaf-disease-classification/test_images"

train_csv_path = "../input/cassava-leaf-disease-classification/train.csv"
train_images_path = "../input/cassava-leaf-disease-classification/train_images"

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(42)



In [3]:
# Change (score improvement): if external .pth is missing, initialize with ImageNet weights.
# This keeps the exact same ResNeXt50 architecture and training loop, but greatly improves
# starting features vs random init, typically increasing accuracy toward the target.
imagenet_weights = models.ResNeXt50_32X4D_Weights.IMAGENET1K_V2
model = models.resnext50_32x4d(
    weights=imagenet_weights if not os.path.exists(model_path) else None
)
model.fc = nn.Linear(2048, 5)
model = model.to(device)

weights_loaded = False
if model_path is not None and os.path.exists(model_path):
    state = torch.load(model_path, map_location=device)
    model.load_state_dict(state)
    weights_loaded = True

print(f"Using device: {device}")
print(f"External pretrained weights loaded: {weights_loaded}")
print(f"Using ImageNet init fallback: {not weights_loaded}")




Downloading: "https://download.pytorch.org/models/resnext50_32x4d-1a0047aa.pth" to /root/.cache/torch/hub/checkpoints/resnext50_32x4d-1a0047aa.pth
100%|██████████| 95.8M/95.8M [00:00<00:00, 112MB/s]


Using device: cuda:0
External pretrained weights loaded: False
Using ImageNet init fallback: True


In [4]:
class CassavaDataset(Dataset):
    def __init__(self, df, img_dir, transform=None, with_labels=True):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
        self.with_labels = with_labels

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, row["image_id"])
        img = np.array(Image.open(img_path).convert("RGB"))
        if self.transform is not None:
            img = self.transform(image=img)["image"]
        img = transforms.ToTensor()(img)
        if self.with_labels:
            y = int(row["label"])
            return img, y
        return img, row["image_id"]


train_df = pd.read_csv(train_csv_path)

# Change (score improvement): use a stratified split to match label distribution in train/valid.
# This reduces variance and makes the short training more reliable (helps accuracy).
train_df = train_df.sample(frac=1.0, random_state=42).reset_index(drop=True)
val_frac = 0.1
tr_parts, va_parts = [], []
for lbl, g in train_df.groupby("label"):
    g = g.sample(frac=1.0, random_state=42)
    n_val = max(1, int(round(len(g) * val_frac)))
    va_parts.append(g.iloc[:n_val])
    tr_parts.append(g.iloc[n_val:])
tr_df = (
    pd.concat(tr_parts, axis=0).sample(frac=1.0, random_state=42).reset_index(drop=True)
)
va_df = (
    pd.concat(va_parts, axis=0).sample(frac=1.0, random_state=42).reset_index(drop=True)
)

train_aug = A.Compose(
    [
        A.RandomResizedCrop(
            size=(256, 256), scale=(0.7, 1.0), ratio=(0.75, 1.3333333333), p=1.0
        ),
        A.HorizontalFlip(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=20, p=0.5),
        A.HueSaturationValue(
            hue_shift_limit=20, sat_shift_limit=20, val_shift_limit=20, p=0.3
        ),
        A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=0.3),
        A.Normalize(
            mean=(0.485, 0.456, 0.406),
            std=(0.229, 0.224, 0.225),
            max_pixel_value=255.0,
            p=1.0,
        ),
    ],
    p=1.0,
)

valid_aug = A.Compose(
    [
        A.Resize(256, 256),
        A.Normalize(
            mean=(0.485, 0.456, 0.406),
            std=(0.229, 0.224, 0.225),
            max_pixel_value=255.0,
            p=1.0,
        ),
    ],
    p=1.0,
)

train_ds = CassavaDataset(
    tr_df, train_images_path, transform=train_aug, with_labels=True
)
valid_ds = CassavaDataset(
    va_df, train_images_path, transform=valid_aug, with_labels=True
)

train_loader = DataLoader(
    train_ds,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)
valid_loader = DataLoader(
    valid_ds,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)

if not weights_loaded:
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

    model.train()
    # Change (score improvement): slightly more epochs is a minimal training-loop change
    # that improves fit from a good initialization; still within time budget due to faster inference later.
    epochs = 5
    for epoch in range(epochs):
        running_loss = 0.0
        correct = 0
        total = 0
        for xb, yb in train_loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * xb.size(0)
            preds = logits.argmax(dim=1)
            correct += (preds == yb).sum().item()
            total += xb.size(0)

        train_loss = running_loss / max(total, 1)
        train_acc = correct / max(total, 1)

        model.eval()
        v_correct = 0
        v_total = 0
        with torch.no_grad():
            for xb, yb in valid_loader:
                xb = xb.to(device, non_blocking=True)
                yb = yb.to(device, non_blocking=True)
                logits = model(xb)
                preds = logits.argmax(dim=1)
                v_correct += (preds == yb).sum().item()
                v_total += xb.size(0)
        val_acc = v_correct / max(v_total, 1)
        print(
            f"Epoch {epoch+1}/{epochs} - train_loss: {train_loss:.4f} - train_acc: {train_acc:.4f} - val_acc: {val_acc:.4f}"
        )
        model.train()

    model.eval()



/usr/local/lib/python3.11/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


Epoch 1/5 - train_loss: 0.6214 - train_acc: 0.7797 - val_acc: 0.8536
Epoch 2/5 - train_loss: 0.4213 - train_acc: 0.8528 - val_acc: 0.8510
Epoch 3/5 - train_loss: 0.3430 - train_acc: 0.8807 - val_acc: 0.8638
Epoch 4/5 - train_loss: 0.2916 - train_acc: 0.8926 - val_acc: 0.8552
Epoch 5/5 - train_loss: 0.2433 - train_acc: 0.9107 - val_acc: 0.8558


In [5]:
# Change (score improvement): make TTA "test-time safe" by using only resize/normalize + flips.
# Using heavy color/dropout/noise at TTA time typically degrades accuracy.
sub_aug = A.Compose(
    [
        A.Resize(256, 256),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.Normalize(
            mean=(0.485, 0.456, 0.406),
            std=(0.229, 0.224, 0.225),
            max_pixel_value=255.0,
            p=1.0,
        ),
    ],
    p=1.0,
)

sub_aug_noaug = A.Compose(
    [
        A.Resize(256, 256),
        A.Normalize(
            mean=(0.485, 0.456, 0.406),
            std=(0.229, 0.224, 0.225),
            max_pixel_value=255.0,
            p=1.0,
        ),
    ],
    p=1.0,
)



In [6]:
assert os.path.exists(
    sample_sub_path
), f"Missing sample_submission.csv at {sample_sub_path}"
assert os.path.isdir(
    test_images_path
), f"Missing test_images directory at {test_images_path}"

sample_sub = pd.read_csv(sample_sub_path)
print(sample_sub.head())
print("Num test images listed:", len(sample_sub))




         image_id  label
0  1234294272.jpg      4
1  1234332763.jpg      4
2  1234375577.jpg      4
3  1234555380.jpg      4
4  1234571117.jpg      4
Num test images listed: 2676


In [7]:
# Change (score improvement + stability): batched inference (same model/preds) reduces overhead,
# allowing higher-tta/epochs while staying within time limits; no change to evaluation semantics.
class CassavaTestDataset(Dataset):
    def __init__(self, df, img_dir, transform):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        image_id = self.df.iloc[idx]["image_id"]
        img_path = os.path.join(self.img_dir, image_id)
        img = np.array(Image.open(img_path).convert("RGB"))
        try:
            img = self.transform(image=img)["image"]
        except Exception:
            img = sub_aug_noaug(image=img)["image"]
        x = transforms.ToTensor()(img)
        return x, image_id


model.eval()

tta_n = 8  # slightly reduced since we removed harmful aug; keeps compute reasonable
all_image_ids = sample_sub["image_id"].tolist()
id_to_sumlogits = {iid: None for iid in all_image_ids}

test_bs = 64 if torch.cuda.is_available() else 16

with torch.no_grad():
    for t in range(tta_n):
        ds_tta = CassavaTestDataset(sample_sub, test_images_path, transform=sub_aug)
        dl_tta = DataLoader(
            ds_tta,
            batch_size=test_bs,
            shuffle=False,
            num_workers=2,
            pin_memory=torch.cuda.is_available(),
        )
        for xb, image_ids in dl_tta:
            xb = xb.to(device, non_blocking=True)
            logits = model(xb).detach().cpu()
            for k, iid in enumerate(image_ids):
                if id_to_sumlogits[iid] is None:
                    id_to_sumlogits[iid] = logits[k].clone()
                else:
                    id_to_sumlogits[iid] += logits[k]

predictions = []
for iid in all_image_ids:
    avg_logits = id_to_sumlogits[iid] / float(tta_n)
    pred_label = int(torch.argmax(avg_logits).item())
    predictions.append([iid, pred_label])

sub_df = pd.DataFrame(predictions, columns=["image_id", "label"])
sub_df.to_csv("submission.csv", index=False)
print(sub_df.head())
print("Wrote submission.csv with rows:", len(sub_df))

         image_id  label
0  1234294272.jpg      3
1  1234332763.jpg      3
2  1234375577.jpg      1
3  1234555380.jpg      1
4  1234571117.jpg      3
Wrote submission.csv with rows: 2676
